<a href="https://colab.research.google.com/github/GitHubAman2004/stock_price_predictor/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hybrid CNN–LSTM–Attention stock direction model

Predicts whether tomorrow's close will be higher than today's (binary).
This version fixes data leakage, runs on GPU when available, evaluates on a
held-out test set, and adds regularization so the model actually trains
instead of collapsing to the majority class.

In [ ]:
%pip install -q yfinance pandas scikit-learn

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Use the GPU the notebook was allocated (Colab T4) whenever it is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print("Using device:", device)

In [ ]:
class HybridForecastingModel(nn.Module):
    """CNN feature extractor -> LSTM -> self-attention (residual) -> classifier."""

    def __init__(self, n_features=5, cnn_channels=16, hidden_size=32,
                 n_heads=4, dropout=0.2):
        super().__init__()
        self.cnn = nn.Conv1d(n_features, cnn_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(cnn_channels, hidden_size, batch_first=True)
        self.attention = nn.MultiheadAttention(hidden_size, n_heads,
                                               dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)            # (B, features, seq) for Conv1d
        x = self.relu(self.cnn(x))
        x = x.permute(0, 2, 1)            # back to (B, seq, channels)

        x, _ = self.lstm(x)
        attn_out, _ = self.attention(x, x, x)
        x = self.norm(x + attn_out)       # residual + LayerNorm for stable training

        last_day = self.dropout(x[:, -1, :])
        return self.fc(last_day)          # raw logits (BCEWithLogitsLoss)


# Sanity check on dummy input.
_m = HybridForecastingModel().to(device)
print("Output shape:", _m(torch.randn(32, 60, 5, device=device)).shape)

In [ ]:
import yfinance as yf
import pandas as pd

ticker = "AAPL"
print(f"Downloading data for {ticker}...")
df = yf.download(ticker, period="5y", auto_adjust=True)
df = df[["Open", "High", "Low", "Close", "Volume"]]

print("\n--- Data downloaded ---")
print("Total trading days:", len(df))
print(df.head())

In [ ]:
from sklearn.preprocessing import MinMaxScaler

sequence_length = 60
data = df.values.astype("float32")

# Chronological split at the row level BEFORE scaling so the scaler never
# sees future/test data (previously fit_transform ran on the whole dataset).
split_row = int(len(data) * 0.80)

scaler = MinMaxScaler()
scaler.fit(data[:split_row])          # fit on training rows only
scaled = scaler.transform(data)

X, y = [], []
for i in range(sequence_length, len(scaled) - 1):
    X.append(scaled[i - sequence_length:i])
    # MinMax is monotonic, so scaled close preserves the up/down direction.
    y.append(1.0 if scaled[i + 1, 3] > scaled[i, 3] else 0.0)

X = torch.tensor(np.array(X), dtype=torch.float32)
y = torch.tensor(np.array(y), dtype=torch.float32).unsqueeze(1)

print("X shape:", X.shape, "| y shape:", y.shape)

In [ ]:
# Split sequences chronologically so every test window is in the future.
split_idx = int(len(X) * 0.80)
X_train, y_train = X[:split_idx], y[:split_idx]
X_test, y_test = X[split_idx:], y[split_idx:]

batch_size = 32
train_loader = DataLoader(TensorDataset(X_train, y_train),
                          batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test),
                         batch_size=batch_size, shuffle=False)

# Class balance -> pos_weight discourages collapsing to a single class.
pos = (y_train == 1).sum().item()
neg = (y_train == 0).sum().item()
pos_weight = torch.tensor([neg / pos], device=device)

print(f"Train: {len(X_train)} ({len(train_loader)} batches) | "
      f"Test: {len(X_test)} ({len(test_loader)} batches)")
print(f"pos/neg in train: {pos}/{neg} | pos_weight: {neg / pos:.2f}")

In [ ]:
def counts(logits, targets):
    pred = (logits > 0).float()
    tp = ((pred == 1) & (targets == 1)).sum().item()
    fp = ((pred == 1) & (targets == 0)).sum().item()
    fn = ((pred == 0) & (targets == 1)).sum().item()
    tn = ((pred == 0) & (targets == 0)).sum().item()
    return tp, fp, fn, tn

def metrics(tp, fp, fn, tn):
    total = tp + fp + fn + tn
    acc = (tp + tn) / total if total else 0.0
    prec = tp / (tp + fp + 1e-7)
    rec = tp / (tp + fn + 1e-7)
    f1 = 2 * prec * rec / (prec + rec + 1e-7)
    return acc, prec, rec, f1

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum = 0.0
    tp = fp = fn = tn = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss_sum += criterion(logits, yb).item()
        a, b, c, d = counts(logits, yb)
        tp += a; fp += b; fn += c; tn += d
    return loss_sum / len(loader), metrics(tp, fp, fn, tn)

In [ ]:
model = HybridForecastingModel().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 30
print("Starting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    tp = fp = fn = tn = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        a, b, c, d = counts(logits, yb)
        tp += a; fp += b; fn += c; tn += d

    train_loss = running_loss / len(train_loader)
    acc, prec, rec, f1 = metrics(tp, fp, fn, tn)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:>2}/{epochs}] | Loss: {train_loss:.4f} | "
              f"Acc: {acc*100:.1f}% | Prec: {prec:.3f} | "
              f"Rec: {rec:.3f} | F1: {f1:.3f}")

print("--- Training complete ---")

In [ ]:
test_loss, (acc, prec, rec, f1) = evaluate(model, test_loader, criterion)
print("--- Held-out test performance ---")
print(f"Loss: {test_loss:.4f} | Acc: {acc*100:.1f}% | "
      f"Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f}")